# TAHAP 1 - Generate Synthetic Fraud Data

Pada tahap ini kita akan membuat **dataset sintetis** untuk melatih model. Dataset ini mencakup:

1. **Transaksi Normal**: penjualan & refund wajar sesuai pola UMKM
2. **Fraud Beragam**: 3 profil fraud dengan tingkat kesulitan berbeda
   - **Profil A (MENCOLOK)**: refund nominal besar di jam dini hari (mudah tertangkap)
   - **Profil B (SEDANG)**: refund nominal menengah di jam normal (perlu fitur rasio)
   - **Profil C (HALUS)**: refund nominal kecil wajar tapi frekuensi tak normal (perlu fitur frekuensi)

**Penting:** Label is_fraud adalah "kunci jawaban" untuk evaluasi nanti. Model TIDAK PERNAH melihat is_fraud saat training.

##  Import & Konfigurasi

In [1]:
import sqlite3
import os
import uuid
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

#   Konfigurasi  
DB_PATH = "database/local_pos.db"
SCHEMA_PATH = "database/schema_sqlite.sql"

N_NORMAL = 2000                              # jumlah transaksi normal
CASHIERS = [f"CSH-{i:03d}" for i in range(1, 6)]   # 5 kasir
RNG_SEED = 42

NORMAL_HOURS = (8, 21)                       # jam operasional toko
SALE_MIN, SALE_MAX = 10_000, 250_000        # nominal jual wajar UMKM (Rp)

rng = np.random.default_rng(RNG_SEED)       # satu RNG global, reproducible
BASE_TIME = datetime(2026, 5, 1, 8, 0, 0)

print(f"[OK] Konfigurasi:")
print(f"     Database: {DB_PATH}")
print(f"     Jumlah normal: {N_NORMAL:,}")
print(f"     Kasir: {', '.join(CASHIERS)}")

[OK] Konfigurasi:
     Database: database/local_pos.db
     Jumlah normal: 2,000
     Kasir: CSH-001, CSH-002, CSH-003, CSH-004, CSH-005


##  Fungsi Pembantu (Vectorized)

In [2]:
def gen_uuids(n: int) -> np.ndarray:
    """Generate N UUID v4 sekaligus menggunakan numpy random bytes (batch, cepat)."""
    raw = rng.integers(0, 256, size=(n, 16), dtype=np.uint8)
    # Set version=4 dan variant bits sesuai RFC 4122
    raw[:, 6] = (raw[:, 6] & 0x0F) | 0x40
    raw[:, 8] = (raw[:, 8] & 0x3F) | 0x80
    return np.array([str(uuid.UUID(bytes=bytes(row))) for row in raw])


def gen_timestamps_vectorized(n: int, day_lo: int, day_hi: int,
                               hour_lo: int, hour_hi: int) -> np.ndarray:
    """Generate N timestamps vectorized — jauh lebih cepat dari loop _rand_time."""
    days    = rng.integers(day_lo, day_hi,  size=n)
    hours   = rng.integers(hour_lo, hour_hi, size=n)
    minutes = rng.integers(0, 60, size=n)
    seconds = rng.integers(0, 60, size=n)
    # Vectorized: BASE_TIME + offset dalam detik
    offsets = (days.astype('int64') * 86_400
               + hours.astype('int64') * 3_600
               + minutes.astype('int64') * 60
               + seconds.astype('int64'))
    base_ts = np.datetime64(BASE_TIME)
    return base_ts + offsets.astype('timedelta64[s]')


print("[OK] Fungsi pembantu (vectorized) selesai")

[OK] Fungsi pembantu (vectorized) selesai


##  Generate Transaksi Normal (Vectorized)

In [3]:
print("[1/3] Generate transaksi NORMAL...")

n = N_NORMAL

#  Semua kolom di-generate sekaligus — tidak ada Python for-loop
cashiers_normal = rng.choice(CASHIERS, size=n)
types_normal    = rng.choice(["SALE", "REFUND"], size=n, p=[0.92, 0.08])
amounts_normal  = np.round(rng.uniform(SALE_MIN, SALE_MAX, size=n), 2)
ts_normal       = gen_timestamps_vectorized(n, 0, 30,
                                            NORMAL_HOURS[0], NORMAL_HOURS[1])
ids_normal      = gen_uuids(n)

df_normal = pd.DataFrame({
    "id":               ids_normal,
    "cashier_id":       cashiers_normal,
    "timestamp":        ts_normal.astype('datetime64[ms]').astype(str),
    "transaction_type": types_normal,
    "amount":           amounts_normal,
    "is_synced":        np.ones(n, dtype=np.int8),
    "is_fraud":         np.zeros(n, dtype=np.int8),
})

n_sale   = (df_normal["transaction_type"] == "SALE").sum()
n_refund = (df_normal["transaction_type"] == "REFUND").sum()
print(f"      {len(df_normal):,} transaksi normal dibuat")
print(f"      Breakdown: SALE {n_sale:,}, REFUND {n_refund:,}")

[1/3] Generate transaksi NORMAL...
      2,000 transaksi normal dibuat
      Breakdown: SALE 1,822, REFUND 178


##  Generate Fraud Profil A (MENCOLOK)

In [4]:
print("\n[2/3] Generate FRAUD - 3 Profil...")
print("\n  Profil A: MENCOLOK (nominal besar, jam dini hari)")

# Kasir 1 — refund besar beruntun cepat di jam 1 pagi
fraud_cashier_a = CASHIERS[0]
N_A = 30

#  Vectorized: nominal & interval waktu sekaligus
amounts_a  = np.round(rng.uniform(500_000, 2_000_000, size=N_A), 2)
intervals_a = rng.integers(15, 60, size=N_A).astype('int64')  # detik antar transaksi
cumulative_a = np.cumsum(intervals_a)                          # offset kumulatif
base_a = np.datetime64(BASE_TIME + timedelta(days=26, hours=1))
ts_a = base_a + cumulative_a.astype('timedelta64[s]')

df_a = pd.DataFrame({
    "id":               gen_uuids(N_A),
    "cashier_id":       fraud_cashier_a,
    "timestamp":        ts_a.astype('datetime64[ms]').astype(str),
    "transaction_type": "REFUND",
    "amount":           amounts_a,
    "is_synced":        np.ones(N_A, dtype=np.int8),
    "is_fraud":         np.ones(N_A, dtype=np.int8),
})

print(f"    Kasir: {fraud_cashier_a}")
print(f"    Jumlah: {N_A} refund besar (Rp 500K-2M)")
print(f"    Waktu: jam dini hari, beruntun cepat")


[2/3] Generate FRAUD - 3 Profil...

  Profil A: MENCOLOK (nominal besar, jam dini hari)
    Kasir: CSH-001
    Jumlah: 30 refund besar (Rp 500K-2M)
    Waktu: jam dini hari, beruntun cepat


##  Generate Fraud Profil B (SEDANG) & C (HALUS)

In [5]:
print("\n  Profil B: SEDANG (nominal menengah, jam normal, 2 kasir)")

#  Profil B: 25 refund per kasir (CSH-003 & CSH-004) — vectorized
N_B_PER = 25
N_B = N_B_PER * 2

cashiers_b = np.repeat([CASHIERS[2], CASHIERS[3]], N_B_PER)
amounts_b  = np.round(rng.uniform(200_000, 400_000, size=N_B), 2)
ts_b       = gen_timestamps_vectorized(N_B, 20, 30, 10, 18)

df_b = pd.DataFrame({
    "id":               gen_uuids(N_B),
    "cashier_id":       cashiers_b,
    "timestamp":        ts_b.astype('datetime64[ms]').astype(str),
    "transaction_type": "REFUND",
    "amount":           amounts_b,
    "is_synced":        np.ones(N_B, dtype=np.int8),
    "is_fraud":         np.ones(N_B, dtype=np.int8),
})

print(f"    Kasir: {CASHIERS[2]}, {CASHIERS[3]}")
print(f"    Jumlah: {N_B} refund (Rp 200K-400K per kasir)")
print(f"    Waktu: jam kerja normal (10-18)")

print("\n  Profil C: HALUS (nominal kecil/wajar, jam normal, frekuensi tinggi)")

#  Profil C: 3 hari × 12 refund = 36 total — vectorized
DAYS_C = [27, 28, 29]
N_C_PER_DAY = 12
N_C = len(DAYS_C) * N_C_PER_DAY
fraud_cashier_c = CASHIERS[4]

amounts_c = np.round(rng.uniform(40_000, 120_000, size=N_C), 2)

# Buat timestamps: per hari mulai di jam acak, lalu tambah interval menit
ts_c_list = []
for day in DAYS_C:
    start_hour = int(rng.integers(9, 17))
    intervals  = rng.integers(2, 12, size=N_C_PER_DAY).astype('int64') * 60  # detik
    base_day   = np.datetime64(BASE_TIME + timedelta(days=day, hours=start_hour))
    ts_c_list.append(base_day + np.cumsum(intervals).astype('timedelta64[s]'))
ts_c = np.concatenate(ts_c_list)

df_c = pd.DataFrame({
    "id":               gen_uuids(N_C),
    "cashier_id":       fraud_cashier_c,
    "timestamp":        ts_c.astype('datetime64[ms]').astype(str),
    "transaction_type": "REFUND",
    "amount":           amounts_c,
    "is_synced":        np.ones(N_C, dtype=np.int8),
    "is_fraud":         np.ones(N_C, dtype=np.int8),
})

print(f"    Kasir: {fraud_cashier_c}")
print(f"    Jumlah: {N_C} refund kecil (Rp 40K-120K)")
print(f"    Waktu: 3 hari dengan {N_C_PER_DAY} refund per hari (frekuensi tak normal)")


  Profil B: SEDANG (nominal menengah, jam normal, 2 kasir)
    Kasir: CSH-003, CSH-004
    Jumlah: 50 refund (Rp 200K-400K per kasir)
    Waktu: jam kerja normal (10-18)

  Profil C: HALUS (nominal kecil/wajar, jam normal, frekuensi tinggi)
    Kasir: CSH-005
    Jumlah: 36 refund kecil (Rp 40K-120K)
    Waktu: 3 hari dengan 12 refund per hari (frekuensi tak normal)


##  Gabungkan & Simpan ke Database

In [6]:
print("\n[3/3] Menggabungkan & menyimpan ke database...")

#  Gabungkan semua DataFrame sekaligus 
df = pd.concat([df_normal, df_a, df_b, df_c], ignore_index=True)

# Parse timestamp & sort
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

# Pastikan tipe data optimal
df["amount"]   = df["amount"].astype("float32")
df["is_synced"] = df["is_synced"].astype("int8")
df["is_fraud"]  = df["is_fraud"].astype("int8")

# Buat direktori database jika belum ada
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

# Connect & buat schema
conn = sqlite3.connect(DB_PATH)
if os.path.exists(SCHEMA_PATH):
    with open(SCHEMA_PATH, "r", encoding="utf-8") as f:
        conn.executescript(f.read())

#  Simpan dengan method='multi' + chunksize untuk insert batch 
df.to_sql("transactions", conn, if_exists="replace", index=False,
          method="multi", chunksize=500)
conn.close()

print(f"  Database: {DB_PATH}")
print(f"  Tabel: transactions")
print(f"  Total baris: {len(df):,}")
print(f"\nGenerate selesai!")


[3/3] Menggabungkan & menyimpan ke database...
  Database: database/local_pos.db
  Tabel: transactions
  Total baris: 2,116

Generate selesai!


##  Ringkasan & Analisis

In [7]:
# Ringkasan data
n_fraud  = int(df["is_fraud"].sum())
n_normal = len(df) - n_fraud

print(" RINGKASAN DATA")
print(f"\n  Total transaksi: {len(df):,}")
print(f"  - Normal       : {n_normal:,} ({n_normal/len(df)*100:.1f}%)")
print(f"  - Fraud        : {n_fraud:,} ({n_fraud/len(df)*100:.1f}%)")

print("\n  Sebaran fraud per kasir (anti-shortcut):")
fraud_dist = df[df.is_fraud == 1]["cashier_id"].value_counts().sort_index()
for cashier, count in fraud_dist.items():
    print(f"    {cashier}: {count:3d} fraud  (Profil: ", end="")
    if cashier == CASHIERS[0]:
        print("A - Mencolok)")
    elif cashier in (CASHIERS[2], CASHIERS[3]):
        print("B - Sedang)")
    elif cashier == CASHIERS[4]:
        print("C - Halus)")
    else:
        print("Tidak ada)")

print("\n  Jenis transaksi:")
print(f"    SALE  : {(df['transaction_type'] == 'SALE').sum():,}")
print(f"    REFUND: {(df['transaction_type'] == 'REFUND').sum():,}")

print("\n  Nominal (Rp):")
print(f"    Min   : {df['amount'].min():,.0f}")
print(f"    Max   : {df['amount'].max():,.0f}")
print(f"    Mean  : {df['amount'].mean():,.0f}")
print(f"    Median: {df['amount'].median():,.0f}")

print("Data siap untuk Tahap 2 (feature engineering)")


 RINGKASAN DATA

  Total transaksi: 2,116
  - Normal       : 2,000 (94.5%)
  - Fraud        : 116 (5.5%)

  Sebaran fraud per kasir (anti-shortcut):
    CSH-001:  30 fraud  (Profil: A - Mencolok)
    CSH-003:  25 fraud  (Profil: B - Sedang)
    CSH-004:  25 fraud  (Profil: B - Sedang)
    CSH-005:  36 fraud  (Profil: C - Halus)

  Jenis transaksi:
    SALE  : 1,822
    REFUND: 294

  Nominal (Rp):
    Min   : 10,154
    Max   : 1,980,273
    Mean  : 145,652
    Median: 129,066
Data siap untuk Tahap 2 (feature engineering)
